# 04 - Inference demo

Use this notebook after distillation to load the saved student checkpoint and generate responses for a few custom prompts.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


In [ ]:
@dataclass(frozen=True)
class DemoConfig:
    student_dir: Path = Path("../checkpoints/student")
    fallback_model: str = "distilgpt2"
    report_dir: Path = Path("../reports")
    max_new_tokens: int = 120
    num_beams: int = 3


config = DemoConfig()
config.report_dir.mkdir(parents=True, exist_ok=True)
asdict(config)


In [ ]:
def select_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def format_instruction(instruction: str, context: str = "") -> str:
    instruction = " ".join(instruction.split())
    context = " ".join(context.split())
    if context:
        return f"### Instruction:\n{instruction}\n\n### Context:\n{context}\n\n### Response:\n"
    return f"### Instruction:\n{instruction}\n\n### Response:\n"


In [ ]:
device = select_device()
model_source = config.student_dir if (config.student_dir / "config.json").exists() else config.fallback_model

tokenizer = AutoTokenizer.from_pretrained(model_source, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_source).to(device)
model.eval()

{"device": str(device), "model_source": str(model_source)}


In [ ]:
@torch.inference_mode()
def generate(prompt: str) -> str:
    encoded = tokenizer(prompt, return_tensors="pt").to(device)
    generated = model.generate(
        **encoded,
        max_new_tokens=config.max_new_tokens,
        num_beams=config.num_beams,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_tokens = generated[0][encoded["input_ids"].shape[1] :]
    return " ".join(tokenizer.decode(new_tokens, skip_special_tokens=True).split())


In [ ]:
demo_prompts = [
    format_instruction("Explain knowledge distillation in one short paragraph."),
    format_instruction("Give two practical reasons to use a smaller student language model."),
    format_instruction("Summarize what temperature does in soft-label distillation."),
]

demo_rows = [{"prompt": prompt, "response": generate(prompt)} for prompt in demo_prompts]
(config.report_dir / "inference_demo.json").write_text(json.dumps(demo_rows, indent=2) + "\n")
demo_rows
